In [ ]:
def pair_features(h1, h2):
    return torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2], dim=-1)

class BranchMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.logit = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        z = self.net(x)
        s = self.logit(z).squeeze(-1)
        return z, s

In [ ]:
class PoolingComparisonModel(nn.Module):
    def __init__(self, d_model=768, hidden_dim=64, dropout=0.4,
                 pooling_type="attention", tau=0.5, topk=3):
        super().__init__()
        self.d_model = d_model
        self.pair_dim = 4 * d_model
        self.pooling_type = pooling_type

        if pooling_type == "mean":
            self.pool = MeanPooling()
        elif pooling_type == "topk":
            self.pool = TopKPooling(k=topk)
        elif pooling_type == "topk_sim":
            self.pool = TopKSimPooling(k=topk)
        elif pooling_type == "attention":
            self.pool = AttentionPooling(tau=tau)
        else:
            raise ValueError(f"Unknown pooling_type: {pooling_type}")

        self.branch_tk = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_ts = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_thk = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_ths = BranchMLP(self.pair_dim, hidden_dim, dropout)

        self.att_w = nn.Linear(hidden_dim, 1)
        self.fuse_out = nn.Linear(hidden_dim, 1)

    def forward(self, batch):
        t = batch["title_emb"].to(DEVICE)
        th = batch["thumb_emb"].to(DEVICE)
        s = batch["stt_embs"].to(DEVICE)
        s_mask = batch["stt_mask"].to(DEVICE)
        k = batch["kf_embs"].to(DEVICE)
        k_mask = batch["kf_mask"].to(DEVICE)

        if self.pooling_type in ["mean", "topk"]:
            s_t, attn_s_t = self.pool(s, s_mask, None)
            k_t, attn_k_t = self.pool(k, k_mask, None)
            s_th, attn_s_th = self.pool(s, s_mask, None)
            k_th, attn_k_th = self.pool(k, k_mask, None)
        else:
            s_t, attn_s_t = self.pool(s, s_mask, t)
            k_t, attn_k_t = self.pool(k, k_mask, t)
            s_th, attn_s_th = self.pool(s, s_mask, th)
            k_th, attn_k_th = self.pool(k, k_mask, th)

        f_tk = pair_features(t, k_t)
        f_ts = pair_features(t, s_t)
        f_thk = pair_features(th, k_th)
        f_ths = pair_features(th, s_th)

        z_tk, s_tk = self.branch_tk(f_tk)
        z_ts, s_ts = self.branch_ts(f_ts)
        z_thk, s_thk = self.branch_thk(f_thk)
        z_ths, s_ths = self.branch_ths(f_ths)

        Z = torch.stack([z_tk, z_ts, z_thk, z_ths], dim=1)   # (B,4,H)

        self.branch_tau = 1.5

        att_scores = self.att_w(Z).squeeze(-1)               # (B,4)
        alpha = F.softmax(att_scores / self.branch_tau, dim=1)

        z_fuse = torch.sum(alpha.unsqueeze(-1) * Z, dim=1)
        s_fuse = self.fuse_out(z_fuse).squeeze(-1)

        return {
            "branch_logits": {
                "tk": s_tk,
                "ts": s_ts,
                "thk": s_thk,
                "ths": s_ths,
            },
            "fused_logit": s_fuse,
            "branch_attention": alpha,
            "token_attention": {
                "s_t": attn_s_t,
                "k_t": attn_k_t,
                "s_th": attn_s_th,
                "k_th": attn_k_th
            }
        }

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# Utility
# -------------------------
def masked_mean(x, mask):
    # x: [B, L, D], mask: [B, L]
    mask = mask.float()
    denom = mask.sum(dim=1, keepdim=True).clamp(min=1e-6)
    return (x * mask.unsqueeze(-1)).sum(dim=1) / denom

def cosine_attention_pool(seq, query, mask, tau=0.5):
    # seq: [B, L, D], query: [B, D], mask: [B, L]
    seq_n = F.normalize(seq, dim=-1)
    q_n = F.normalize(query, dim=-1).unsqueeze(1)   # [B,1,D]
    sim = (seq_n * q_n).sum(dim=-1)                 # [B,L]
    sim = sim / tau
    sim = sim.masked_fill(mask == 0, -1e9)
    attn = F.softmax(sim, dim=-1)                   # [B,L]
    pooled = torch.bmm(attn.unsqueeze(1), seq).squeeze(1)  # [B,D]
    return pooled, attn

def topk_sim_pool(seq, query, mask, k=3):
    # seq: [B,L,D], query: [B,D], mask: [B,L]
    seq_n = F.normalize(seq, dim=-1)
    q_n = F.normalize(query, dim=-1).unsqueeze(1)
    sim = (seq_n * q_n).sum(dim=-1)                 # [B,L]
    sim = sim.masked_fill(mask == 0, -1e9)

    k = min(k, seq.size(1))
    topk_idx = sim.topk(k=k, dim=-1).indices        # [B,k]

    gathered = torch.gather(
        seq, 1, topk_idx.unsqueeze(-1).expand(-1, -1, seq.size(-1))
    )                                               # [B,k,D]
    pooled = gathered.mean(dim=1)                   # [B,D]
    return pooled

def topk_norm_pool(seq, mask, k=3):
    norm = seq.norm(dim=-1)                         # [B,L]
    norm = norm.masked_fill(mask == 0, -1e9)

    k = min(k, seq.size(1))
    topk_idx = norm.topk(k=k, dim=-1).indices
    gathered = torch.gather(
        seq, 1, topk_idx.unsqueeze(-1).expand(-1, -1, seq.size(-1))
    )
    pooled = gathered.mean(dim=1)
    return pooled

def pair_features(h1, h2):
    return torch.cat([h1, h2, torch.abs(h1 - h2), h1 * h2], dim=-1)


# -------------------------
# Pooling modules
# -------------------------
class MeanPooling(nn.Module):
    def forward(self, seq, query, mask):
        return masked_mean(seq, mask)

class TopKPooling(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.k = k

    def forward(self, seq, query, mask):
        return topk_norm_pool(seq, mask, self.k)

class TopKSimPooling(nn.Module):
    def __init__(self, k=3):
        super().__init__()
        self.k = k

    def forward(self, seq, query, mask):
        return topk_sim_pool(seq, query, mask, self.k)

class AttentionPooling(nn.Module):
    def __init__(self, tau=0.5):
        super().__init__()
        self.tau = tau

    def forward(self, seq, query, mask):
        pooled, _ = cosine_attention_pool(seq, query, mask, tau=self.tau)
        return pooled


# -------------------------
# Branch MLP
# -------------------------
class BranchMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = self.net(x)
        logit = self.out(h).squeeze(-1)
        return h, logit


# -------------------------
# Ablation model
# -------------------------
class ModalityAblationModel(nn.Module):
    """
    active_branches examples:
    ["ts"]
    ["thk"]
    ["tk", "ts", "thk", "ths"]   # All Branches
    """

    def __init__(
        self,
        d_model=768,
        hidden_dim=64,
        dropout=0.4,
        pooling_type="attention",
        tau=0.5,
        topk=3,
        active_branches=("tk", "ts", "thk", "ths")
    ):
        super().__init__()
        self.d_model = d_model
        self.pair_dim = 4 * d_model
        self.active_branches = list(active_branches)

        if pooling_type == "mean":
            self.pool = MeanPooling()
        elif pooling_type == "topk":
            self.pool = TopKPooling(k=topk)
        elif pooling_type == "topk_sim":
            self.pool = TopKSimPooling(k=topk)
        elif pooling_type == "attention":
            self.pool = AttentionPooling(tau=tau)
        else:
            raise ValueError(f"Unknown pooling_type: {pooling_type}")

        self.branch_tk  = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_ts  = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_thk = BranchMLP(self.pair_dim, hidden_dim, dropout)
        self.branch_ths = BranchMLP(self.pair_dim, hidden_dim, dropout)

        self.pair_ln = nn.LayerNorm(self.pair_dim)

        # fusion
        self.att_w = nn.Linear(hidden_dim, 1)
        self.fuse_out = nn.Linear(hidden_dim, 1)

    def encode_pairs(self, batch):
        title = batch["title_emb"]
        thumb = batch["thumb_emb"]
        stt_seq = batch["stt_embs"]
        stt_mask = batch["stt_mask"]
        key_seq = batch["kf_embs"]
        key_mask = batch["kf_mask"]

        stt_by_title = self.pool(stt_seq, title, stt_mask)
        stt_by_thumb = self.pool(stt_seq, thumb, stt_mask)
        key_by_title = self.pool(key_seq, title, key_mask)
        key_by_thumb = self.pool(key_seq, thumb, key_mask)

        pairs = {
            "tk": self.pair_ln(pair_features(title, key_by_title)),
            "ts": self.pair_ln(pair_features(title, stt_by_title)),
            "thk": self.pair_ln(pair_features(thumb, key_by_thumb)),
            "ths": self.pair_ln(pair_features(thumb, stt_by_thumb)),
        }
        return pairs

    def forward(self, batch):
        pairs = self.encode_pairs(batch)

        branch_outputs = {}
        used_h = []
        used_logits = []

        if "tk" in self.active_branches:
            h, logit = self.branch_tk(pairs["tk"])
            branch_outputs["tk"] = logit
            used_h.append(h)
            used_logits.append(logit)

        if "ts" in self.active_branches:
            h, logit = self.branch_ts(pairs["ts"])
            branch_outputs["ts"] = logit
            used_h.append(h)
            used_logits.append(logit)

        if "thk" in self.active_branches:
            h, logit = self.branch_thk(pairs["thk"])
            branch_outputs["thk"] = logit
            used_h.append(h)
            used_logits.append(logit)

        if "ths" in self.active_branches:
            h, logit = self.branch_ths(pairs["ths"])
            branch_outputs["ths"] = logit
            used_h.append(h)
            used_logits.append(logit)

        if len(used_h) == 0:
            raise ValueError("No active branches selected.")

        # only selected branches participate in fusion
        H = torch.stack(used_h, dim=1)                 # [B, n_branch, hidden]
        alpha = torch.softmax(self.att_w(H).squeeze(-1), dim=1)  # [B, n_branch]
        fused = (H * alpha.unsqueeze(-1)).sum(dim=1)  # [B, hidden]
        fuse_logit = self.fuse_out(fused).squeeze(-1)

        return {
            "branch_logits": branch_outputs,
            "fuse_logit": fuse_logit,
            "alpha": alpha
        }